# DiffCAM in NumPy with handcrafted spatial features

**Weeks:** 30 · **Core question:** How does a feature-space direction become a spatial saliency map?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

We now connect spatial channels to pooled embeddings without training a CNN. Each channel is a deterministic image transformation, and its mean is a feature. The reference group determines weights; weights then combine the *target* spatial feature maps.

This is an educational reproduction of DiffCAM's core calculation using handcrafted channels, not a reproduction of published image results.

**Learning goal:** Explain each step from `[N,C,H,W]` maps to target-relative saliency.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from scipy.ndimage import gaussian_filter, sobel
from diffcam_edu import make_cell_images
from diffcam_edu.math import diffcam_weights,diffcam_map,normalize_map
x,y,m=make_cell_images(100,seed=6)
def channels(image):
    return np.stack([image, gaussian_filter(image,1),
                     np.abs(sobel(image,axis=0)),np.abs(sobel(image,axis=1))])
A=np.stack([channels(im) for im in x])
z=A.mean(axis=(-2,-1))
target=int(np.flatnonzero(y==1)[0]);refs=z[y==0]
weights=diffcam_weights(z[target],refs,ridge=.1)
print('Spatial maps:',A.shape,'Embeddings:',z.shape,'Weights:',weights.round(3))

### Experiment 2 of 3

In [ ]:
signed=diffcam_map(A[target],weights,positive_only=False)
positive=diffcam_map(A[target],weights,positive_only=True)
fig,ax=plt.subplots(1,3,figsize=(10,3))
for a,img,title in zip(ax,[x[target],signed,positive],['Cell','Signed weighted map','Positive-only map']):
    a.imshow(img,cmap='coolwarm' if title.startswith('Signed') else 'inferno');a.set_title(title);a.axis('off')
plt.show()

### Experiment 3 of 3

In [ ]:
from diffcam_edu.evaluation import area_under_precision_recall
score=area_under_precision_recall(positive,m[target])
print('Region AUPRC (illustrative):',score)
assert positive.shape==x[target].shape

## Exercises — complete independently
1. Change the reference set from class 0 to class 1 and compare both weight vectors and saliency maps.
2. Add a fifth spatial feature: an intensity threshold map. Does it overwhelm the existing channels?

## Check your understanding

**Exit ticket:** Explain each step from `[N,C,H,W]` maps to target-relative saliency.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)